<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# Finite State Machine

Your Duckiebot runs many nodes at once: line detectors, lane filters, controllers, LED emitters, and more.
Not all of them should be active all of the time. When you grab the joystick, lane following should stop.
When an obstacle appears, the robot should brake. The **Finite State Machine (FSM)** node is the central
coordinator that decides *which nodes are active* at any given moment and *how the robot reacts to events*.

Concretely, when the state changes the FSM node:

1. enables or disables other nodes by calling their `switch` services, and
2. optionally changes the LED pattern.
3. decides which controller should be connected to the wheels

In this notebook we will build a small, runnable model of the FSM node so you can *see* how a configuration
file turns into behavior, fire events at it, and watch it transition between states, all without a robot.

## How it works

At startup, the FSM node:

1. Loads its configuration from a YAML file (via ROS params).
2. Waits for all declared `nodes` to advertise their `/switch` services.
3. Subscribes to all declared `events` topics.
4. Publishes the initial state on `~mode` and activates the corresponding nodes.

When an event arrives (a `BoolStamped` message on a watched topic), the FSM checks whether its value matches
the event's `trigger`. If it does, it looks up the next state, **first** in the current state's `transitions`,
**then** in `global_transitions`, and if a transition exists it switches to the new state, calls each node's
switch service, and updates the LEDs.

> **Transition priority:** per-state `transitions` take precedence over `global_transitions` for the same event name.

The current state is published on `~mode` as a latched `FSMState` message, and you can also set the state
manually through the `~set_state` service:

```bash
rosservice call /[robot]/fsm_node/set_state "state: 'LANE_FOLLOWING'"
```

## Anatomy of a configuration file

FSM config files live in `dt-core/packages/fsm/config/fsm_node/`. Each one is a YAML document with these
top-level keys:

| Key | Required | Meaning |
|-----|----------|---------|
| `initial_state` | yes | The state the FSM starts in on boot. Must be a key under `states`. |
| `nodes` | yes | Map from a logical node name to the ROS `switch` service the FSM calls to enable/disable it. |
| `events` | yes | Map from event name to the topic + trigger value that fires it. |
| `states` | yes | Map from state name to its `active_nodes`, optional `transitions`, and optional `lights`. |
| `global_transitions` | no | Transitions that apply in *every* state unless a per-state transition overrides them. |

Below is the **real** `lane_following.yaml` config that ships with this learning experience. Run the cell to
load it, we will drive the FSM with it in a moment.

In [ ]:
import yaml
from pprint import pprint

# This is the actual contents of
#   dt-core/packages/fsm/config/fsm_node/lane_following.yaml
LANE_FOLLOWING_YAML = """
initial_state: "NORMAL_JOYSTICK_CONTROL"

events:                       # Maps a subscribed topic + trigger value to an event name
  joystick_override_on:
    topic: "joy_mapper_node/joystick_override"
    msg_type: "BoolStamped"
    trigger: True
  joystick_override_off:
    topic: "joy_mapper_node/joystick_override"
    msg_type: "BoolStamped"
    trigger: False

nodes:                        # Logical node name -> ROS switch service
  anti_instagram: "anti_instagram_node/switch"
  line_detector_node: "line_detector_node/switch"
  lane_filter_node: "lane_filter_node/switch"
  ground_projection_node: "ground_projection_node/switch"
  lane_controller_node: "lane_controller_node/switch"
  led_emitter_node: "led_emitter_node/switch"

global_transitions:
  joystick_override_on: "NORMAL_JOYSTICK_CONTROL"
  joystick_override_off: "LANE_FOLLOWING"

states:
  NORMAL_JOYSTICK_CONTROL:
    active_nodes:
      - anti_instagram
      - lane_filter_node
      - line_detector_node
      - ground_projection_node
      - led_emitter_node
    lights: "GREEN"
  LANE_FOLLOWING:
    active_nodes:
      - anti_instagram
      - line_detector_node
      - lane_filter_node
      - ground_projection_node
      - lane_controller_node
      - led_emitter_node
    lights: "CAR_DRIVING"
"""

config = yaml.safe_load(LANE_FOLLOWING_YAML)
print("States defined:", list(config["states"].keys()))
print("Events defined:", list(config["events"].keys()))
print("Initial state:", config["initial_state"])

## A minimal FSM, in Python

The class below mirrors the logic of the real `fsm_node`, minus the ROS plumbing. Instead of calling
`switch` services over the network, it just records which nodes *would* be enabled. The two pieces of logic
worth paying attention to are:

* **`validate()`** runs the same sanity checks the real node runs on startup, it refuses to start with a
  broken config.
* **`fire()`** implements the transition lookup with per-state transitions taking priority over global ones.

In [ ]:
class FSM:
    def __init__(self, config):
        self.config = config
        self.states = config["states"]
        self.events = config["events"]
        self.nodes = config["nodes"]
        self.global_transitions = config.get("global_transitions", {})
        self.validate()
        self.state = config["initial_state"]

    def validate(self):
        """The same checks the real fsm_node runs on startup."""
        if self.config["initial_state"] not in self.states:
            raise ValueError(f"initial_state {self.config['initial_state']!r} is not a defined state")
        for name, ev in self.events.items():
            for field in ("topic", "msg_type", "trigger"):
                if field not in ev:
                    raise ValueError(f"event {name!r} is missing required field {field!r}")
        # every next_state referenced must exist
        all_transitions = list(self.global_transitions.items())
        for sname, sdef in self.states.items():
            all_transitions += list(sdef.get("transitions", {}).items())
        for event_name, next_state in all_transitions:
            if next_state not in self.states:
                raise ValueError(f"transition on {event_name!r} points at undefined state {next_state!r}")
        print("Config is valid.")

    @property
    def active_nodes(self):
        return self.states[self.state].get("active_nodes", [])

    @property
    def lights(self):
        return self.states[self.state].get("lights")

    def fire(self, event_name):
        """Try to take the transition for event_name from the current state."""
        if event_name not in self.events:
            raise ValueError(f"unknown event {event_name!r}")
        # per-state transitions take priority over global_transitions
        per_state = self.states[self.state].get("transitions", {})
        if event_name in per_state:
            next_state, source = per_state[event_name], "state"
        elif event_name in self.global_transitions:
            next_state, source = self.global_transitions[event_name], "global"
        else:
            print(f"  [{event_name}] ignored in state {self.state} (no matching transition)")
            return self.state
        prev = self.state
        self.state = next_state
        print(f"  [{event_name}] {prev} --({source})--> {self.state}")
        return self.state

    def describe(self):
        print(f"State:  {self.state}")
        print(f"Lights: {self.lights}")
        print("Active nodes:")
        for n in self.nodes:
            mark = "on " if n in self.active_nodes else "off"
            print(f"  [{mark}] {n}")

Let's boot the FSM with the lane following config and look at the initial state.
Notice which nodes are active in `NORMAL_JOYSTICK_CONTROL`, the `lane_controller_node` is **off**, so the
robot will not try to drive itself until we transition into `LANE_FOLLOWING`.

In [ ]:
fsm = FSM(config)
fsm.describe()

## Driving the state machine

Now let's fire some events. In the real system these come from `BoolStamped` messages, e.g.
`joy_mapper_node/joystick_override` going `True` produces the `joystick_override_on` event. Here we just
call them by name.

Both of these transitions are **global**, so they fire no matter which state we are in. That is exactly what
you want for a safety override: pressing the joystick should *always* drop you back into manual control.

In [ ]:
print("Press the joystick (override on):")
fsm.fire("joystick_override_on")

print("\nRelease the joystick (override off) -> autonomous lane following:")
fsm.fire("joystick_override_off")

print()
fsm.describe()

## Who actually drives? The `car_cmd_switch_node`

Enabling a node with `active_nodes` is only half the story. In `LANE_FOLLOWING` the `lane_controller_node`
is on and computing wheel commands, but in other states a *different* controller (the joystick, an
intersection controller, ...) should be steering, and several of these controllers can even be running at
once. So how does the robot decide *whose* commands actually reach the motors?

That decision is made by a dedicated node, **`car_cmd_switch_node`**, and it is driven by the FSM. Recall
that the FSM publishes the current state on `~mode` (a latched `FSMState` message). The
`car_cmd_switch_node` subscribes to that exact topic (`fsm_node/mode`) and uses the state to select **one**
command source to forward downstream to the wheels.

Its config (`robots/duckiebot/dagu_car/config/car_cmd_switch_node/default.yaml`) has two maps:

* `source_topics` — named command sources → the topic each one publishes its `car_cmd` on
  (e.g. `lane` → `lane_controller_node/car_cmd`, `joystick` → `joy_mapper_node/car_cmd`).
* `mappings` — FSM state → which source name is in control in that state. Several states can map to the
  same source (e.g. every intersection state maps to `stop`).

So the FSM state acts as a **multiplexer select line**:

```
fsm_node/mode  →  mappings[state]  →  source_topics[name]  →  the one car_cmd stream sent to the wheels
```

The special `stop` source is a safety shortcut: instead of forwarding any controller, the node publishes a
zero-velocity command, an immediate brake, no matter what the controllers are saying. This is why the
`active_nodes` list and the command routing are complementary: the FSM both *enables the right nodes* and,
through `car_cmd_switch_node`, *routes the right one to the motors*.

In [ ]:
# The two maps from car_cmd_switch_node/default.yaml
SOURCE_TOPICS = {
    "lane":         "lane_controller_node/car_cmd",
    "intersection": "unicorn_intersection_node/car_cmd",
    "coordination": "coordinator_node/car_cmd",
    "joystick":     "joy_mapper_node/car_cmd",
    "stop":         "simple_stop_controller_node/car_cmd",
}
MODE_TO_SOURCE = {
    "NORMAL_JOYSTICK_CONTROL":    "joystick",
    "LANE_FOLLOWING":             "lane",
    "DETECT_INTERSECTION_TYPE":   "stop",
    "STOP_SIGN_INTERSECTION":     "stop",
    "TRAFFIC_LIGHT_INTERSECTION": "stop",
    "INTERSECTION_CONTROL":       "intersection",
    "STOP":                       "stop",
}

def who_is_driving(state):
    src = MODE_TO_SOURCE.get(state)
    if src is None:
        return f"{state}: no source selected -> no command forwarded to the wheels"
    if src == "stop":
        return f"{state}: source 'stop' -> robot is braked (zero velocity)"
    return f"{state}: source '{src}' -> forwards {SOURCE_TOPICS[src]} to the wheels"

# In the two states our lane-following FSM uses, the switch routes joystick vs lane controller:
for state in ["NORMAL_JOYSTICK_CONTROL", "LANE_FOLLOWING"]:
    print(who_is_driving(state))

# ...and for whichever state the FSM is in right now (after the transitions above):
print("\nFSM is currently in:", fsm.state)
print(who_is_driving(fsm.state))

## Visualizing the state machine

A picture makes the structure much easier to read. The cell below renders the states as nodes and the
transitions as labeled edges (dashed = global transition). It uses `graphviz` if it is installed, and falls
back to a plain-text listing otherwise, so it runs anywhere.

In [ ]:
def draw(fsm):
    edges = []  # (src, dst, label, is_global)
    for sname, sdef in fsm.states.items():
        for ev, dst in sdef.get("transitions", {}).items():
            edges.append((sname, dst, ev, False))
    for ev, dst in fsm.global_transitions.items():
        for sname in fsm.states:
            # a per-state transition on the same event overrides the global one
            if ev not in fsm.states[sname].get("transitions", {}):
                edges.append((sname, dst, ev, True))
    try:
        from graphviz import Digraph
        g = Digraph()
        g.attr(rankdir="LR")
        for sname in fsm.states:
            shape = "box"
            g.node(sname, shape=shape, style="rounded,filled", fillcolor="#fff3c4")
        for src, dst, label, is_global in edges:
            g.edge(src, dst, label=label, style="dashed" if is_global else "solid")
        return g
    except Exception as e:
        print(f"(graphviz not available: {e})\nText fallback:\n")
        for src, dst, label, is_global in edges:
            tag = "global" if is_global else "state"
            print(f"  {src:24s} --{label} [{tag}]--> {dst}")
        return None

draw(fsm)

## Writing your own configuration

Now that you can see how a config drives behavior, here is the field-by-field guide for writing your own.

### `nodes`
A map from a logical node name (used inside `active_nodes` lists) to the ROS service path the FSM calls to
enable/disable it. The FSM calls each service with `True` when the node appears in the current state's
`active_nodes`, and `False` otherwise. **Every node you want to control must be listed here**, even if it is
only active in one state.

### `events`
A map from event name to the topic and trigger value that fires it. Only `BoolStamped` messages are
supported. The same topic can appear in two events with opposite `trigger` values (e.g.
`joystick_override_on` / `joystick_override_off`).

### `states`
Each state has `active_nodes` (the nodes enabled in this state, all others disabled), an optional
`transitions` map (event name -> next state), and an optional `lights` LED pattern. A state with no
`transitions` key is a terminal state, it can only be exited via a `global_transition`.

### `global_transitions`
Transitions that apply in **every** state unless a per-state `transitions` entry overrides them for the same
event. Useful for safety events like joystick override.

### Validation rules
The FSM validates the config on startup and shuts down if any of these are violated:

* Every `next_state` referenced in `transitions` or `global_transitions` must exist in `states`.
* Every event must have `topic`, `msg_type`, and `trigger` fields.
* `initial_state` must match a key in `states`.

The `validate()` method above enforces exactly these rules. Let's prove it catches a mistake.

In [ ]:
broken = yaml.safe_load(LANE_FOLLOWING_YAML)
# Point a transition at a state that doesn't exist:
broken["global_transitions"]["joystick_override_off"] = "FLYING"

try:
    FSM(broken)
except ValueError as e:
    print("Rejected, as expected:")
    print(" ", e)

## Exercise: add an obstacle-stop behavior

Extend the config so the robot brakes when it sees an obstacle and resumes when the path is clear. To do
this you will:

1. Add two events, `obstacle_detected` and `obstacle_cleared`, on a `BoolStamped` topic.
2. Add a new `STOP` state where `lane_controller_node` is **off** (so the robot does not drive) but the
   perception nodes stay on. Give it `lights: "RED"`.
3. Add a **per-state** transition from `LANE_FOLLOWING` on `obstacle_detected` to `STOP`, and from `STOP`
   on `obstacle_cleared` back to `LANE_FOLLOWING`.

Fill in the `TODO`s below and run the cell. If `validate()` passes and the diagram shows the new `STOP`
state wired in, you've got it.

In [ ]:
my_config = yaml.safe_load(LANE_FOLLOWING_YAML)

# 1. add the events
my_config["events"]["obstacle_detected"] = {
    "topic": "tof_obstacle_detection_node/obstacle_detected",
    "msg_type": "BoolStamped",
    "trigger": True,
}
# TODO: add an 'obstacle_cleared' event (same topic, trigger: False)

# 2. add the STOP state
my_config["states"]["STOP"] = {
    "active_nodes": [
        "anti_instagram",
        "line_detector_node",
        "lane_filter_node",
        "ground_projection_node",
        "led_emitter_node",
        # note: lane_controller_node is intentionally absent -> robot stops
    ],
    "lights": "RED",
    # TODO: add a 'transitions' map: obstacle_cleared -> LANE_FOLLOWING
}

# 3. wire LANE_FOLLOWING -> STOP
# TODO: give LANE_FOLLOWING a 'transitions' entry: obstacle_detected -> STOP

my_fsm = FSM(my_config)
draw(my_fsm)

Once your config is complete, this should walk all the way through: start in joystick control, begin lane
following, hit an obstacle, stop, clear it, and resume.

In [ ]:
my_fsm = FSM(my_config)
for ev in ["joystick_override_off", "obstacle_detected", "obstacle_cleared"]:
    if ev in my_fsm.events:
        my_fsm.fire(ev)
    else:
        print(f"  [{ev}] not defined yet -- finish the exercise above")
print()
my_fsm.describe()

## Recap

* The FSM node is the **coordinator**: a state decides which nodes are switched on and what the LEDs do.
* **Events** (`BoolStamped` topics + a trigger value) drive **transitions** between states.
* Per-state `transitions` beat `global_transitions`, use globals for safety overrides that should fire
  anywhere.
* The config is validated on startup; a dangling state reference or a malformed event stops the node from
  launching.

Now we have written all the code we need in order for the robot to be able to boot up in `NORMAL_JOYSTICK_CONTROL` mode and then we can trigger the `LANE_FOLLOWING` mode by publishing a `BoolStamped` variable `joystick_override_off=True` which incidently is exactly what happens when we toggle the `AutoPilot` switch with the keyboard controller. 

In the [next and final notebook](./06_building_the_demo.ipynb), we will describe how to construct all of this into an executable that can be run with the `dts devel` API as we saw at the beginning. 